# Audit of missing judgments versus explicit relevance 0

This notebook checks whether the old construction fallback
`annotation_lookup.get((abstract_id, aspect_id), 0)` was ever activated.

It audits both:

1. all constructed aspect rows; and
2. the exact training subset remaining after zero-positive rows are removed.

A score of zero counts as **explicit** only when the `(abstract_id, aspect_id)`
key is present in the original annotation table. A missing key is counted
separately, regardless of the score stored in the constructed dataset.

In [ ]:
import hashlib
import json
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "DorisMAE" / "Annotation.json").exists() and (
            candidate / "HRIA" / "reranking_dataset_doris_mae.jsonl"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root containing DorisMAE/Annotation.json "
        "and HRIA/reranking_dataset_doris_mae.jsonl."
    )


REPO_ROOT = find_repo_root(Path.cwd())
ANNOTATION_PATH = REPO_ROOT / "DorisMAE" / "Annotation.json"
CONSTRUCTED_DATASET_PATH = (
    REPO_ROOT / "HRIA" / "reranking_dataset_doris_mae.jsonl"
)
REPORT_PATH = (
    REPO_ROOT / "New folder" / "missing_relevance_fallback_audit.json"
)

print(f"Repository root: {REPO_ROOT}")
print(f"Annotations: {ANNOTATION_PATH}")
print(f"Constructed dataset: {CONSTRUCTED_DATASET_PATH}")

In [ ]:
from collections import Counter


def build_annotation_lookup(annotation_records):
    """Build the original last-write lookup and report duplicate labels."""
    lookup = {}
    duplicate_count = 0
    conflicts = []

    for record in annotation_records:
        key = (int(record["abstract_id"]), str(record["aspect_id"]))
        score = int(record["score"])

        if key in lookup:
            duplicate_count += 1
            if lookup[key] != score:
                conflicts.append({
                    "abstract_id": key[0],
                    "aspect_id": key[1],
                    "first_score": lookup[key],
                    "conflicting_score": score,
                })

        # Match pandas Series.to_dict(), used by the original constructor:
        # when a key repeats, the last record determines the stored score.
        lookup[key] = score

    return lookup, duplicate_count, conflicts


def audit_rows(rows, annotation_lookup, max_examples=20):
    """Audit raw annotation membership independently of stored fallback scores."""
    score_counts = Counter()
    missing_examples = []
    score_mismatch_examples = []
    missing_pairs = 0
    candidate_aspect_pairs = 0

    for row_index, row in enumerate(rows):
        aspect_id = str(row["aspect_id"])
        candidate_pool = row["candidate_pool"]
        stored_scores = row.get("candidate_scores")

        if stored_scores is not None and len(stored_scores) != len(candidate_pool):
            raise ValueError(
                f"Row {row_index} has {len(candidate_pool)} candidates but "
                f"{len(stored_scores)} stored scores."
            )

        for position, abstract_id in enumerate(candidate_pool):
            abstract_id = int(abstract_id)
            key = (abstract_id, aspect_id)
            candidate_aspect_pairs += 1

            if key not in annotation_lookup:
                missing_pairs += 1
                if len(missing_examples) < max_examples:
                    missing_examples.append({
                        "row_index": row_index,
                        "position": position,
                        "abstract_id": abstract_id,
                        "aspect_id": aspect_id,
                        "stored_score": (
                            None
                            if stored_scores is None
                            else int(stored_scores[position])
                        ),
                    })
                continue

            annotation_score = int(annotation_lookup[key])
            score_counts[annotation_score] += 1

            if stored_scores is not None:
                stored_score = int(stored_scores[position])
                if stored_score != annotation_score and len(
                    score_mismatch_examples
                ) < max_examples:
                    score_mismatch_examples.append({
                        "row_index": row_index,
                        "position": position,
                        "abstract_id": abstract_id,
                        "aspect_id": aspect_id,
                        "stored_score": stored_score,
                        "annotation_score": annotation_score,
                    })

    return {
        "rows": len(rows),
        "candidate_aspect_pairs": candidate_aspect_pairs,
        "missing_pairs": missing_pairs,
        "explicit_zero_pairs": score_counts[0],
        "positive_pairs": sum(
            count for score, count in score_counts.items() if score > 0
        ),
        "score_counts": {
            str(score): count for score, count in sorted(score_counts.items())
        },
        "missing_examples": missing_examples,
        "score_mismatch_examples": score_mismatch_examples,
    }

In [ ]:
print("Loading original annotations...")
with ANNOTATION_PATH.open("r", encoding="utf-8") as file:
    annotation_records = json.load(file)

(
    annotation_lookup,
    duplicate_annotation_count,
    conflicting_annotations,
) = build_annotation_lookup(annotation_records)
del annotation_records

print("Loading constructed aspect rows...")
with CONSTRUCTED_DATASET_PATH.open("r", encoding="utf-8") as file:
    all_rows = [json.loads(line) for line in file if line.strip()]

training_rows = [
    row
    for row in all_rows
    if any(int(score) > 0 for score in row["candidate_scores"])
]

print(f"Unique annotation keys: {len(annotation_lookup):,}")
print(f"Repeated annotation records: {duplicate_annotation_count:,}")
print(f"Conflicting duplicate keys: {len(conflicting_annotations):,}")
if conflicting_annotations:
    print(json.dumps(conflicting_annotations[:20], indent=2))
print(f"All constructed rows: {len(all_rows):,}")
print(f"Rows reaching training: {len(training_rows):,}")

In [ ]:
all_rows_audit = audit_rows(all_rows, annotation_lookup)
training_rows_audit = audit_rows(training_rows, annotation_lookup)

print("\nAll constructed rows:")
print(json.dumps(all_rows_audit, indent=2))
print("\nExact training subset:")
print(json.dumps(training_rows_audit, indent=2))

In [ ]:
def sha256_file(path: Path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


report = {
    "audit_name": "missing_relevance_fallback_audit",
    "definition": (
        "A pair is missing only when (abstract_id, aspect_id) is absent "
        "from the original Annotation.json lookup."
    ),
    "annotation_path": str(ANNOTATION_PATH.relative_to(REPO_ROOT)),
    "annotation_sha256": sha256_file(ANNOTATION_PATH),
    "constructed_dataset_path": str(
        CONSTRUCTED_DATASET_PATH.relative_to(REPO_ROOT)
    ),
    "constructed_dataset_sha256": sha256_file(CONSTRUCTED_DATASET_PATH),
    "unique_annotation_keys": len(annotation_lookup),
    "repeated_annotation_records": duplicate_annotation_count,
    "conflicting_duplicate_keys": len(conflicting_annotations),
    "conflicting_annotation_examples": conflicting_annotations[:20],
    "all_constructed_rows": all_rows_audit,
    "training_subset": training_rows_audit,
    "fallback_triggered": all_rows_audit["missing_pairs"] > 0,
}

REPORT_PATH.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved audit report to: {REPORT_PATH}")

assert not all_rows_audit["score_mismatch_examples"], (
    "Constructed scores disagree with original annotations: "
    f"{all_rows_audit['score_mismatch_examples'][:5]}"
)
assert all_rows_audit["missing_pairs"] == 0, (
    "The relevance-zero fallback was triggered. Missing examples: "
    f"{all_rows_audit['missing_examples'][:5]}"
)

print("AUDIT PASSED: the missing-to-zero fallback was never triggered.")

## Interpretation

If the final assertion passes, every score-zero training label is backed by an
explicit record in `Annotation.json`. The old `.get(..., 0)` fallback remains
fragile and should be replaced with explicit missing-value handling, but it did
not affect this instantiated dataset.